# Calibration of RAMA: a pyramid WFS, and its deformable mirror.

RAMA is a Pyramid-based AO system installed in the 0.6 m FEELINGS telescope in Toulouse, France. It has an Alpao97 DM

The calibration procedure is as follows:
1. We load the data, and pre-process it to facilitate the calibration.
2. We make a first rough calibration to place the pupils in the correct positions
3. We calibrate the static amplitude and phase from the bench
4. We define the deformable mirror and perform a rough alignment to determine rotations, flips and signs
5. Using the fully differentiable WFS and DM models we fit all the degrees of freedom to compute the misregistration

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'


from AI4AO import PyramidWFS, DeformableMirror, imshow, TwinCalibrator

from mmengine import Config
import numpy as np
import matplotlib.pyplot as plt

You can download the data to calibrate from here

[Link to files](https://nuage.osupytheas.fr/s/d9SqPNAYeq8Lkgc)

We load the bench interaction matrix and pupil map, and the modes-to-commands matrix `M2C`. RAMA's pyramid produces four separate pupil images; we stitch them into a single 2x2-tiled frame with `TwinCalibrator.tile_pyramid_frame` to work with them as one image, and derive a binary reference frame from the interaction matrix's pixel-wise standard deviation (we don't have a dedicated bench reference frame for RAMA).

In [ ]:
imat_data = np.load(r"../../Data/Rama/Rama_imats2.npz")
iMat_modal = imat_data["iMat_modal"]
iMat_zonal = imat_data["iMat_zonal"]
pyrMap = imat_data["pupils"]

M2C = np.load(r"../../Data/Rama/M2C.npy")
# M2C = np.eye(iMat_zonal.shape[0])
M2C = torch.from_numpy(M2C).to(device = device, dtype = torch.float32)


# _COORDS = ((24, 30), (529, 32), (530, 524), (31, 530))
_COORDS = ((27, 31), (529, 31), (529, 527), (27, 527))
pupil_size = 36
pupil_separation = 6
pyrImSizeX = 560
pyrImSizeY = 560

iMatCrop = TwinCalibrator.tile_pyramid_frame(
    iMat_modal, pyrMap, _COORDS, (pyrImSizeX, pyrImSizeY), pupil_size, pupil_separation
)

iMat_bench = torch.from_numpy(iMatCrop).to(device=device, dtype = torch.float32)

reference_frame = iMat_bench.std(dim = 0)
reference_frame = (reference_frame > reference_frame.max() * 0.02).to(device=device, dtype = torch.float32)
reference_frame /= reference_frame.sum()

In [ ]:
WFSParams = dict(
    {
        "Nres":34,
        "sampling":96/34,
        "D": 0.6,
        "useNoise": False,
        "centralObstruction": 0.,
        "Modulation": 3,
        "Wavelength": 635e-9
    }
)

In [ ]:
wfs = PyramidWFS(WFSParams, device)
PATH_WFS = "../Data/Rama/RamaWFS.pth"
# try:
#     wfs.LoadCalibration(PATH_WFS)
# except:
#     print("Starting from scratch")

calibrator = TwinCalibrator(wfs, dm=None, device=device)

## Pupil positioning

We start the calibration by placing the pupils in the correct positions, using `TwinCalibrator.fit_pupil_to_reference`. It:
1. Updates the position of the pupils with `wfs.BuildMask()`
2. Computes a reference intensity with `wfs.BuildReferenceIntensity()`
3. Compares it to the reference frame and backpropagates

It is possible that the pupils are not aligned on the first try and you can run the cell as many times as you need. Also, you can change `n_iter` to have more or less iterations, and `lr` to change the learning rate of the optimizer.

In [ ]:
final_loss = calibrator.fit_pupil_to_reference(
    reference_frame,
    wfs.parameters(),
    lr=2e-3,
    n_iter=200,
    loss_fn=lambda ref, dig: torch.nn.MSELoss()(ref * 1e5, dig * 1e5),
)

## DM parameters

`moffatParam`: Corresponds to the moffat exponent which interpolates between a Cauchy and Gaussian shapes for the influence functions.

`signedAmplitude`: Amplitude in OPD of the DM. Can be positive or negative, depending on convention.

`Flips`: Flip left-right or top-bottom. If you don't know them, don't worry as there is a rough calibration step that can find the best configuration.

`misreg`: Dictionary containing the misregistrations of the DM. These are compatible with OOPAO's misreg.

If you don't know the correct flips, rotations, and sign of the DM, `calibrator.rough_calibrate_dm(bench_iMat, M2C)` performs an automatic selection of the best starting values.

In [ ]:
DMParams = dict(
    {
        "Nactuator": 11,
        "Nmodes": 2,
        "moffatParam": 2,
        "signedAmplitude": -1e-6,
        "MechCoupling": 0.36,
        "FlipLeftRight": False,
        "FlipTopBottom": False
    }
)

dm = DeformableMirror(WFSParams,
                    DMParams,
                    device=device,
                    offset_to_fit_number_of_actuators=0.1)

calibrator.dm = dm
calibrator.rough_calibrate_dm(iMat_bench, M2C)

We define the static amplitude and phase maps for the bench. These will get optimized along with the misreg of the DM. To avoid overfitting or falling into local minima, the training is set such that these maps are not updated at the beginning and slowly start getting more and more importance as the optimization progresses.

In [ ]:
ref_pupil, ref_phase = calibrator.init_static_offsets()

Always check on some modes to see if the rough alignment makes sense! You can change `idx` to see through some modes.

In [ ]:
index = list(range(0,96,1))

calibrator.sanity_check_plot(iMat_bench, M2C, index, idx=3)

In [ ]:
# Align the WFS pupil mask to the DM's illuminated aperture before the joint fit below
wfs.pupil = torch.clone(dm.pupil)

## Optimization of misregistration, WFS parameters and static amplitude and phase

We define the optimization problem to be: find the best parameters for the DM, WFS and offsets to produce a synthetic interaction matrix as close as possible to the one measured on the bench, using `calibrator.fit_dm_and_offsets`.

Given that the main parameters to be trained are the DM's, the learning rate is set higher than for the WFS, which was previously optimized. For the static amplitude and phase, to avoid overfitting, the optimization starts with an extremely low learning rate for them, so the optimizer prioritizes changing the parameters of the DM and WFS first. The learning rate for the offsets is then gradually increased to match that of the DM and WFS.

In [ ]:
final_loss, original_positons, transformed_positons = calibrator.fit_dm_and_offsets(
    iMat_bench,
    M2C,
    index,
    n_iter=300,
    lr_dm=1e-2,
    lr_wfs=1e-3,
    lr_offset_start=-10,
    lr_offset_end=-2,
    fit_static_offsets=True,
    plot_mode_idx=4,
)

You can check the retrieved actuator positions, the static amplitude and phase values.

In [ ]:
calibrator.plot_actuator_and_offsets(original_positons, transformed_positons)

We now compute a new interaction matrix with the fitted parameters.

In [ ]:
modes = calibrator.rebuild_reconstruction_matrix(M2C)

You can change the index to see how well the algorithm worked to fit the parameters.

In [ ]:
idx = 57
target_idx = iMat_bench[idx].cpu().detach().numpy()
calibrator.plot_fit_residual(
    iMat_bench, idx, vmin=target_idx.min(), vmax=target_idx.max(), reshape_digital=True
)

In [ ]:
cov = calibrator.crosstalk_diagnostic(iMat_bench)

In [ ]:
PATH_WFS, PATH_DM = calibrator.save("Rama", data_dir="../../Data")

In [ ]:
wfs = PyramidWFS(WFSParams, device)
dm = DeformableMirror(WFSParams,
                    DMParams,
                    device=device)

calibrator.wfs = wfs
calibrator.dm = dm
calibrator.load("Rama", data_dir="../../Data")

modes = calibrator.rebuild_reconstruction_matrix(M2C)
cov = calibrator.crosstalk_diagnostic(iMat_bench)

Finally, we convert the fitted synthetic interaction matrix back from our 2x2-tiled layout into the bench's original flat valid-pixel format with `TwinCalibrator.untile_pyramid_image`, so it can be compared to or exported alongside the raw bench data.

In [ ]:
synth_iMat = wfs.iMat.cpu().detach().numpy()

synth_iMat_to_bench = np.zeros_like(iMat_modal)

synth_iMat_full = np.zeros((pyrImSizeX, pyrImSizeY, synth_iMat.shape[0]))
for i in range(synth_iMat.shape[0]):
    test = TwinCalibrator.untile_pyramid_image(
        synth_iMat[i],
        _COORDS,
        full_shape=(pyrImSizeX, pyrImSizeY),
        pupil_size=pupil_size,
        pupil_separation=pupil_separation)
    
    synth_iMat_to_bench[i] = test[pyrMap != 0]

# np.save("synth_iMat_v2.npy", synth_iMat_to_bench)